# EmoDial on Colab

Runtime → Change runtime type → **T4 GPU**. Outputs are saved to Google Drive so a disconnect does not lose trained models.

Each member can run only the step they own; every step reads the previous step's files from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/emodial_runs'   # shared outputs live here
os.makedirs(WORK, exist_ok=True)

In [ ]:
# Clone the repo (private repo: use a GitHub token, see README)
REPO = 'https://github.com/denzillll/emodial.git'
!rm -rf /content/emodial && git clone -q $REPO /content/emodial
%cd /content/emodial
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Point data/ and outputs/ at Drive so every step persists
!mkdir -p $WORK/data $WORK/outputs
!rm -rf data outputs && ln -s $WORK/data data && ln -s $WORK/outputs outputs
!ls -la data outputs

## 1 · Data (Stevson)

In [ ]:
!python scripts/01_prepare_data.py

## 2 · Emotion recogniser (Denzil)

In [ ]:
!python scripts/02_train_recogniser.py
!cat outputs/recogniser/metrics.json | head -40

## 3 · Cache p_t / Δ_t + Δ statistics (Stevson)
Check `with_previous_user_turn`, the shift quantiles, `tau` and the ED sanity check.

In [ ]:
!python scripts/03_cache_emotions.py
!cat data/cache/delta_stats.json

## 4 · Generator: shared base checkpoint, then the five systems (York Khang)
One variant ≈ 1 h on a T4. Add `data.max_train_examples=2000` to any command for a quick debug run.

In [ ]:
!python scripts/04_train_generator.py --stage base

In [ ]:
SEED = 42
for v in ['b0', 'b1', 'etg_nodelta', 'etg_nogate', 'etg']:
    !python scripts/04_train_generator.py --variant {v} --seed {SEED}

## 5 · Generate replies on the fixed test subset

In [ ]:
for v in ['b0', 'b1', 'etg_nodelta', 'etg_nogate', 'etg']:
    !python scripts/05_generate.py --variant {v} --seed {SEED} --swap

## 6 · Automatic evaluation (Mikolaj)

In [ ]:
!python scripts/06_evaluate.py
!cat outputs/results/summary.md

## 7 · Human evaluation sheets (Aleksandra)
Share `outputs/human_eval/ratings_<name>.csv` and `pairwise_<name>.csv` with each rater. **Never share the KEY file.** When everyone has filled theirs in, put them back in the same folder and run `analyse`.

In [ ]:
!python scripts/07_human_eval.py make
# later:
# !python scripts/07_human_eval.py analyse